<a href="https://colab.research.google.com/github/ksprashu/gemma4-finetuning-cloudrun/blob/main/gemma4_colab_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Gemma 4 Fine-Tuning & Inference in Google Colab (T4 GPU)

This companion notebook walks you step-by-step through setting up, testing, and fine-tuning Google's **Gemma 4 (Edge 2B)** model using standard **Hugging Face Transformers, PEFT, and TRL** on a free **NVIDIA T4 GPU** inside Google Colab.

### 🧠 What is Gemma 4 E2B?
The **Gemma 4 E2B** (Edge 2B) is part of Google's lightweight open-weights model family, engineered specifically for edge and mobile environments. It supports native multimodal inputs and fits comfortably inside memory-constrained devices. It comes in two primary forms:
1. **`google/gemma-4-E2B` (Base Model)**: Pre-trained on a massive corpus. It excels at autocompleting text but doesn't understand conversational instruction formats.
2. **`google/gemma-4-E2B-it` (Instruction-Tuned Model)**: Fine-tuned by Google to follow conversational formatting, reply to user prompts, and act as an interactive assistant.

### 🎭 Model Behavioral Comparison Matrix
This matrix displays the exact behavior expected from each model variant given the **exact same prompt**:

| Model Variant | Expected Behavior | Example Output | Linguistic/Statistical Explanation |
| :--- | :--- | :--- | :--- |
| **Instruction-Tuned (IT)**<br>`google/gemma-4-E2B-it` | **Task Aligned Helper**<br>Understands conversational framing, targets system prompts, and responds succinctly. | `Positive` | It has been aligned using **Supervised Fine-Tuning (SFT)** and RLHF to identify conversational markers (e.g. user-assistant templates) and output task compliance. |
| **Base Model**<br>`google/gemma-4-E2B` | **Raw Statistical Autocomplete**<br>Ignores the command and treats the prompt as the start of a document, continuing to list reviews or templates. | `Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'`<br>`Classify the sentiment: 'Bad battery.'`<br>`Classify the sentiment: 'Excellent stay.'` | It acts as a **high-entropy token completer**. It only knows how to continue the statistical pattern of the input text, not how to follow an instruction. |
| **Fine-Tuned Model**<br>`Base Model + Sentiment Adapter` | **Domain Aligned Specialist**<br>Constrains its high-entropy autocomplete states to output *only* your target sentiment label. | `Positive` | QLoRA backpropagation updated the adapter's linear projection layers, steering attention heads to emit only the targeted classification tokens when prompted with the instruction template. |

### ⚡ QLoRA: Fine-Tuning 2B parameters on a T4 GPU
- **4-bit Quantization**: We load the base model in 4-bit precision (NF4), reducing the base memory footprint to **~1.5GB to 2.0GB VRAM**.
- **LoRA Adapters**: Instead of updating all 2B parameters, we freeze the base model and train small, lightweight matrices (adapters) attached to the target layers (`q_proj`, `v_proj`, etc.).
- **Paged 8-bit Optimizers**: We use `paged_adamw_8bit` which offloads optimizer memory states to system RAM during training peaks.

This notebook is **100% self-contained** and can be run end-to-end on Colab's free tier!

---


In [1]:
# Install required packages for training, quantization, and GCS integrations
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets google-cloud-storage


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.8 MB/s eta 0:00:00


## 🔑 Step 1: Hugging Face Authentication

Because Gemma 4 is a **gated model**, you must accept the license terms on the Hugging Face model cards before running this notebook:
- [Gemma 4 E2B Base](https://huggingface.co/google/gemma-4-E2B)
- [Gemma 4 E2B IT](https://huggingface.co/google/gemma-4-E2B-it)

Once accepted, grab your **Hugging Face Token** and enter it below. The cell is designed to automatically check for Colab's built-in **Secrets** manager (recommended) or fall back to an interactive login block.

> [!IMPORTANT]
> **🔑 Sharing and Pushing to HF Hub:** If you plan to upload your fine-tuned model weights to the **Hugging Face Hub** at the end of this notebook, make sure to generate and use a Hugging Face Token with **Write** permissions (from your [Hugging Face Settings -> Access Tokens](https://huggingface.co/settings/tokens)). A read-only token can download the models but will fail during push operations.


In [2]:
import os
try:
    from google.colab import userdata
    # If you have added HF_TOKEN as a Colab Secret (the left key-icon menu), load it directly
    hf_token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HF_TOKEN detected and loaded from Colab Secrets!")
except Exception:
    print("ℹ️ HF_TOKEN not found in Colab Secrets. Falling back to interactive login...")
    from huggingface_hub import notebook_login
    notebook_login()


✅ HF_TOKEN detected and loaded from Colab Secrets!


## 💬 Step 2: Testing the Instruction-Tuned (IT) Model

We will load `google/gemma-4-E2B-it` in **4-bit quantization** to demonstrate how an aligned conversational model behaves. It will follow your prompt's system instructions and output a clean, formatted sentiment label.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

it_model_id = "google/gemma-4-E2B-it"

# 1. Define 4-bit Quantization (QLoRA config)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for T4 compatibility
    bnb_4bit_use_double_quant=True
)

# 2. Load the Model & Tokenizer
print(f"⏳ Loading instruction-tuned model '{it_model_id}' in 4-bit...")
it_model = AutoModelForCausalLM.from_pretrained(
    it_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Unwrap Gemma4ClippableLinear modules if present to prevent PEFT/LoRA errors
for name, module in list(it_model.named_modules()):
    if module.__class__.__name__ == 'Gemma4ClippableLinear':
        parts = name.split('.')
        parent = it_model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], module.linear)

it_tokenizer = AutoTokenizer.from_pretrained(it_model_id)
print("✅ Instruction-Tuned model loaded successfully!")

# 3. Test prompt using standard Chat Template
messages = [
    {"role": "user", "content": "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'"}
]

formatted_prompt = it_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = it_tokenizer(formatted_prompt, return_tensors="pt").to(it_model.device)

print(f"\n[Formatted Prompt for Model]:\n{formatted_prompt}\n")

with torch.no_grad():
    outputs = it_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=it_tokenizer.eos_token_id
    )

prompt_len = inputs.input_ids.shape[1]
generated_ids = outputs[0][prompt_len:]
response = it_tokenizer.decode(generated_ids, skip_special_tokens=True)

print("\n============================================================")
print("🌟 IT MODEL INFERENCE RESULTS & BEHAVIORAL ANALYSIS")
print("============================================================")
print(f"Original Prompt: \"Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'\"")
print(f"Raw Generated Text:\n{response.strip()}")
print("-"*60)
print("💡 ANALYST EXPLANATION:")
print("The Instruction-Tuned (IT) model understands turn-taking syntax because")
print("it has undergone SFT and RLHF alignment. It recognizes user queries inside")
print("conversational templates and behaves as an assistant, targeting your")
print("instruction directly and returning the classification label or structured answer.")
print("============================================================")


### 🎮 Try Your Own Prompts on the IT Model!
Before we clear the model from memory to avoid running out of RAM, use the interactive form-field below to test other prompts of your choice.


In [ ]:
#@title 💬 Interactive IT Model Playground { run: "auto" }
#@markdown Type your own prompt below to test how the aligned instruction-tuned model responds.

custom_prompt = "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'" #@param {type:"string"}

# Ensure model is still in memory
if 'it_model' in globals() and 'it_tokenizer' in globals():
    messages = [{"role": "user", "content": custom_prompt}]
    formatted_prompt = it_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = it_tokenizer(formatted_prompt, return_tensors="pt").to(it_model.device)

    with torch.no_grad():
        outputs = it_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=it_tokenizer.eos_token_id
        )

    prompt_len = inputs.input_ids.shape[1]
    response = it_tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()

    print("="*60)
    print("🎭 IT MODEL RESPONSE:")
    print("="*60)
    print(response)
    print("="*60)
else:
    print("⚠️ Error: The Instruction-Tuned model has already been deleted from VRAM. Please re-run the loading cell above.")


### 🧹 Freeing VRAM Memory
Because T4 GPUs are limited, we must delete the loaded IT model and clear PyTorch's cache before loading the Base model to prevent out-of-memory crashes.


In [ ]:
import sys
import gc
import torch

# 1. Clear model, tokenizer, and input references from the namespace
for var in ['it_model', 'it_tokenizer', 'inputs', 'outputs', 'formatted_prompt', 'custom_prompt', 'response']:
    if var in globals():
        del globals()[var]

# 2. Clear traceback references to release memory held by errors
sys.last_traceback = None
sys.last_value = None
sys.last_type = None

# 3. Flush IPython's command history / Out dict cache
try:
    ipython = get_ipython()
    if ipython is not None:
        ipython.user_ns.pop('_', None)
        ipython.user_ns.pop('__', None)
        ipython.user_ns.pop('___', None)
        for key in list(ipython.user_ns.keys()):
            if key.startswith('_') and key[1:].isdigit():
                ipython.user_ns.pop(key, None)
        if 'Out' in ipython.user_ns:
            ipython.user_ns['Out'].clear()
except NameError:
    pass

# 4. Trigger garbage collector & empty PyTorch CUDA cache
gc.collect()
torch.cuda.empty_cache()
print("🧹 VRAM memory cleared! 100% reference cache flushed successfully.")


## 📝 Step 3: Testing the Base Model (Contrastive Autocomplete)

Now, we will load the pre-trained base model `google/gemma-4-E2B` and run the **exact same prompt**.

Observe the output: instead of answering with a classification label, the base model will continue autocompleting reviews or creating similar templates. This is normal base model behavior because it lacks conversational instruction-alignment.


In [ ]:
base_model_id = "google/gemma-4-E2B"

print(f"⏳ Loading base model '{base_model_id}' in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Unwrap Gemma4ClippableLinear modules if present to prevent PEFT/LoRA errors
for name, module in list(base_model.named_modules()):
    if module.__class__.__name__ == 'Gemma4ClippableLinear':
        parts = name.split('.')
        parent = base_model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], module.linear)

base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
print("✅ Base model loaded successfully!")

# Run the exact same prompt
prompt = "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'"
inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=base_tokenizer.eos_token_id
    )

response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n============================================================")
print("📝 BASE MODEL INFERENCE RESULTS & BEHAVIORAL ANALYSIS")
print("============================================================")
print(f"Original Prompt: \"{prompt}\"")
print(f"Raw Generated Text:\n{response}")
print("-"*60)
print("⚠️ ANALYST EXPLANATION:")
print("Observe how the Base Model completely ignored the command 'Classify the sentiment'!")
print("Instead of outputting 'Positive', it autocompleted our text. It likely added")
print("more hypothetical reviews (e.g. 'Classify the sentiment: Bad battery...', etc.).")
print("This high-entropy document completion is the standard, expected behavior")
print("of an unaligned base model, which acts as a pure statistical text completer.")
print("============================================================")


### 🎮 Try Your Own Prompts on the Base Model!
Use the interactive form below to test other custom prompts on the Base Model. Notice how it behaves as a pure autocomplete engine, continuing whatever pattern you start.


In [ ]:
#@title 📝 Interactive Base Model Playground { run: "auto" }
#@markdown Type your own text prompt below to observe how the unaligned base model autocompletes/continues it.

custom_prompt = "Classify the sentiment: 'I had an absolutely wonderful experience!'" #@param {type:"string"}

if 'base_model' in globals() and 'base_tokenizer' in globals():
    inputs = base_tokenizer(custom_prompt, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=base_tokenizer.eos_token_id
        )

    response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("="*60)
    print("📝 BASE MODEL AUTOCOMPLETE RESPONSE:")
    print("="*60)
    print(response)
    print("="*60)
else:
    print("⚠️ Error: The Base model has not been loaded yet, or was removed. Please run the Base model loading cell above.")


### 🧹 Freeing VRAM Memory Before Training
To avoid running out of memory during fine-tuning, we must delete our Step 3 inference base model and clear PyTorch's cache. This ensures the GPU has maximum free VRAM for gradient updates and optimizer states during training.


In [ ]:
import sys
import gc
import torch

# 1. Clear model, tokenizer, and input references from the namespace
for var in ['base_model', 'base_tokenizer', 'inputs', 'outputs', 'custom_prompt', 'response']:
    if var in globals():
        del globals()[var]

# 2. Clear traceback references to release memory held by errors
sys.last_traceback = None
sys.last_value = None
sys.last_type = None

# 3. Flush IPython's command history / Out dict cache
try:
    ipython = get_ipython()
    if ipython is not None:
        ipython.user_ns.pop('_', None)
        ipython.user_ns.pop('__', None)
        ipython.user_ns.pop('___', None)
        for key in list(ipython.user_ns.keys()):
            if key.startswith('_') and key[1:].isdigit():
                ipython.user_ns.pop(key, None)
        if 'Out' in ipython.user_ns:
            ipython.user_ns['Out'].clear()
except NameError:
    pass

# 4. Trigger garbage collector & empty PyTorch CUDA cache
gc.collect()
torch.cuda.empty_cache()
print("🧹 VRAM memory cleared and ready for fine-tuning!")


## 📊 Step 4: Programmatic & Agentic Dataset Synthesis

To prepare your Gemma 4 model for training, you have two major pathways for dataset preparation:

### 🧱 Pathway A: Local Programmatic Script (Included Below)
To make this notebook **100% self-contained**, we include a local python cell that programmatically compiles **500 sentiment analysis samples** covering varied linguistic nuances (such as double negations, slang, sarcasm, and emoji emphasis) and saves them locally to `sentiment_dataset.jsonl` matching Gemma's conversational training schema:
```json
{
  "messages": [
    {"role": "user", "content": "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'"},
    {"role": "model", "content": "Positive"}
  ]
}
```

---

### 🚀 Pathway B: Agentic Synthesis with Google Antigravity (AGY)
While programmatic template generation is useful for a quick demo, it has highly limited vocabulary diversity. For production-grade fine-tuning, you need high-fidelity synthetic data. By using the **AGY CLI (`agy`)** or **Antigravity 2.0 Desktop app**, you can command the AI to act as a **Producer Archetype** to generate thousands of unique, natural, and complex samples with structured schema constraints.

#### 💡 Other Domain Ideas & Precise AGY Prompts
Here are three other high-value domain-specific use cases you can fine-tune Gemma 4 on, along with the exact prompts you can copy and paste directly into **AGY** to generate your custom datasets:

##### 1️⃣ Multilingual Support Ticket Router
* **Concept:** Fine-tune Gemma to act as an automated ticket triage gateway, classifying raw customer messages (English, Spanish, French, German, Japanese, Hindi) into issues (`billing`, `tech_support`, `account_security`, `refund_request`) and priorities (`critical`, `high`, `medium`, `low`).
* **AGY Prompt:**
  ```
  Role: Act as a data generation engineer (Producer Archetype).
  Task: Create a synthetic fine-tuning dataset of 1000 customer support triage samples.
  Constraints:
  - The input support ticket should be in random languages (English, Spanish, French, German, Japanese, Hindi).
  - The model must output a JSON block indicating category and priority.
  - Format: Save as a JSON Lines (.jsonl) file in my current workspace named `triage_dataset.jsonl`.
  - Schema: Every line must be a single JSON object matching this structure exactly:
    {"messages": [{"role": "user", "content": "Triage this support ticket: '<TICKET_TEXT>'"}, {"role": "model", "content": "{\"category\": \"<CAT>\", \"priority\": \"<PRIORITY>\"}"}]}
  ```

##### 2️⃣ Text-to-API Payload Copilot (Function Caller)
* **Concept:** Fine-tune Gemma to translate unstructured natural language request intents into structured REST API JSON payloads for backend microservices.
* **AGY Prompt:**
  ```
  Role: Act as a data generation engineer (Producer Archetype).
  Task: Create a synthetic fine-tuning dataset of 1000 Text-to-API translation samples.
  Constraints:
  - The user prompt should be a natural language request to perform an action (e.g., booking, profile updates).
  - The model response should be a formatted, structured REST API payload.
  - Format: Save as a JSON Lines (.jsonl) file in my current workspace named `api_dataset.jsonl`.
  - Schema: Every line must be a single JSON object matching this structure exactly:
    {"messages": [{"role": "user", "content": "Translate to API payload: '<REQUEST_TEXT>'"}, {"role": "model", "content": "{\"action\": \"<ACTION>\", \"params\": {<PARAMETERS>}}"}]}
  ```

##### 3️⃣ Enterprise PII Redactor
* **Concept:** Fine-tune Gemma to redact/mask sensitive customer information (emails, names, SSNs, credit card numbers, phone numbers) before logging chat logs.
* **AGY Prompt:**
  ```
  Role: Act as a data generation engineer (Producer Archetype).
  Task: Create a synthetic fine-tuning dataset of 1000 PII anonymization samples.
  Constraints:
  - The input should be a conversational transcript containing names, credit card numbers, email addresses, phone numbers, or SSNs.
  - The model response must be the exact same transcript, but with all PII replaced by standardized tags like `[REDACTED_NAME]`, `[REDACTED_EMAIL]`, `[REDACTED_PHONE]`, `[REDACTED_CARD]`.
  - Format: Save as a JSON Lines (.jsonl) file in my current workspace named `pii_dataset.jsonl`.
  - Schema: Every line must be a single JSON object matching this structure exactly:
    {"messages": [{"role": "user", "content": "Redact PII from this transcript: '<TRANSCRIPT>'"}, {"role": "model", "content": "<REDACTED_TRANSCRIPT>"}]}
  ```

Let's execute Pathway A to programmatically synthesize our sentiment dataset and run the training process.


In [3]:
import json
import random

# Define lexicon databases for synthesis
domains = ["product", "movie", "app", "restaurant", "stay"]
templates = {
    "pos": [
        "This {domain} is incredible and highly recommended.",
        "Absolutely loved my stay, everything was spotless!",
        "It works like a charm, worth every single dollar.",
        "I don't dislike this {domain} at all, it's amazing.",
        "This is easily the GOAT! Extremely top-tier 🔥",
        "Although the setup was steep, it is spectacular!"
    ],
    "neg": [
        "Worst {domain} I have ever spent money on.",
        "Oh great, another bug-filled disaster... exactly what I needed.",
        "It broke down after only three days of light usage.",
        "Avoid at all costs. Absolute waste of time.",
        "Cheap material, clunky interface, and sluggish support.",
        "I really wanted to support this, but it is a complete trainwreck."
    ]
}

synthetic_data = []
for i in range(500):
    sentiment = random.choice(["Positive", "Negative"])
    tpl_group = "pos" if sentiment == "Positive" else "neg"
    raw_review = random.choice(templates[tpl_group]).format(domain=random.choice(domains))

    # Randomize instruction wrapping
    inst = f"Classify the sentiment: '{raw_review}'"

    synthetic_data.append({
        "messages": [
            {"role": "user", "content": inst},
            {"role": "model", "content": sentiment}
        ]
    })

# Save locally to file
dataset_path = "sentiment_dataset.jsonl"
with open(dataset_path, "w", encoding="utf-8") as f:
    for sample in synthetic_data:
        f.write(json.dumps(sample) + "\n")

print(f"✅ Successfully synthesized 500 samples in: '{dataset_path}'")


✅ Successfully synthesized 500 samples in: 'sentiment_dataset.jsonl'


## 🏋️ Step 5: Fine-Tuning Gemma Base with QLoRA

Now we will load our dataset into the Hugging Face `SFTTrainer` and execute our QLoRA training run on the T4 GPU.

> [!IMPORTANT]
> **💥 CRITICAL: Google Colab T4 Memory Management & Session Restart Protocol**
> Because we loaded and played with both the **Instruction-Tuned** and **Base** models in Step 2 and Step 3, Python's garbage collector and Google Colab's interactive forms may have lingering tensor references in memory. Even minor VRAM leaks of 1-2 GB will cause an `OutOfMemoryError` during training because `prepare_model_for_kbit_training` and gradient updates require every megabyte of our 15GB VRAM.
>
> **🔄 To ensure a 100% successful training run with ZERO memory issues:**
> 1. Go to the top menu and select **Runtime -> Restart session** (or use shortcut `Ctrl + M` then `.`).
>    * *Note: Restarting the session clears the Python state and frees 100% of VRAM, but **keeps** all pip installations and files intact!*
> 2. Once the session is restarted, skip the loading cells in Step 2 & 3, and run **Step 1** (HF Authentication) to load your credentials.
> 3. Proceed directly to **Step 5** below to begin fine-tuning!

### How the adapters are applied:
1. We freeze the loaded 4-bit base model `google/gemma-4-E2B` to prevent its original parameters from shifting.
2. We define a `LoraConfig` that targets only the projection modules. This inserts low-rank matrices into the network, isolating trainable params.
3. We trigger `SFTTrainer` which runs sequence batching, computes loss, and updates our adapter matrices.


In [4]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

# 1. Load the generated dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"Dataset Loaded. Record Count: {len(dataset)}")

# 2. Reload Base Model & Tokenizer Fresh for Fine-Tuning
base_model_id = "google/gemma-4-E2B"
print(f"⏳ Loading fresh base model '{base_model_id}' in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for T4 compatibility
    bnb_4bit_use_double_quant=True
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16, # Load unquantized weights in float16 to prevent bfloat16 mixed-type GradScaler errors on T4
    device_map="auto"
)

# Unwrap Gemma4ClippableLinear modules if present to prevent PEFT/LoRA errors
for name, module in list(base_model.named_modules()):
    if module.__class__.__name__ == 'Gemma4ClippableLinear':
        parts = name.split('.')
        parent = base_model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], module.linear)

base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
print("✅ Fresh Base model loaded successfully!")

# 3. Custom memory-efficient prepare_model_for_kbit_training
def custom_prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs=None):
    for name, param in model.named_parameters():
        param.requires_grad = False

    # Only upcast tiny normalization layers (norm, ln) to float32 for stability.
    # We skip upcasting the massive 2.8B parameter embedding & language model head layers,
    # saving ~8.75 GiB of VRAM from being allocated!
    for name, param in model.named_parameters():
        if ("norm" in name or "ln" in name) and param.__class__.__name__ != "Params4bit":
            if param.dtype in [torch.float16, torch.bfloat16]:
                param.data = param.data.to(torch.float32)

    # Explicitly convert any remaining bfloat16 parameters and buffers to float16 (essential for T4 GPU compatibility)
    for name, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
    for name, buf in model.named_buffers():
        if buf.dtype == torch.bfloat16:
            buf.data = buf.data.to(torch.float16)

    if use_gradient_checkpointing:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gradient_checkpointing_kwargs)
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
        else:
            def make_inputs_require_grad(module, input, output):
                output.requires_grad_(True)
            model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)
    return model

base_model = custom_prepare_model_for_kbit_training(base_model)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "right"

# Set standard Gemma chat template since we are using a conversational messages dataset
base_tokenizer.chat_template = (
    "{{ bos_token }}"
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ '<start_of_turn>user\\n' + message['content'] + '<end_of_turn>\\n' }}"
    "{% elif message['role'] == 'model' %}"
    "{{ '<start_of_turn>model\\n' + message['content'] + '<end_of_turn>\\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<start_of_turn>model\\n' }}"
    "{% endif %}"
)

# 4. Configure LoRA parameters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

# 5. Define modern TRL SFT Training Parameters
output_dir = "./results"
training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=1, # 1 Epoch is enough for this self-contained demo
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    optim="paged_adamw_8bit", # Saves optimizer memory
    fp16=True, # Use FP16 for standard T4 operations
    bf16=False,
    logging_steps=10,
    save_strategy="no",
    max_length=512,
    report_to="none"
)

# 6. Initialize SFTTrainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=base_tokenizer,
    args=training_args
)

# Explicitly align parameter dtypes for mixed-precision training on T4 GPU:
# 1. Trainable weights (LoRA adapters) must remain in float32 for GradScaler gradient accumulation.
# 2. Frozen weights (embeddings, head) must be cast from bfloat16 to float16 to prevent bfloat16 activation leakage.
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        if param.dtype != torch.float32:
            param.data = param.data.to(torch.float32)
    else:
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
for name, buf in trainer.model.named_buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

# 7. Execute Fine-Tuning!
print("⏳ Starting QLoRA fine-tuning...")
trainer.train()
print("✅ Fine-Tuning complete!")

# 8. Save Adapter locally
adapter_dir = "./fine_tuned_gemma_adapter"
trainer.model.save_pretrained(adapter_dir)
base_tokenizer.save_pretrained(adapter_dir)
print(f"💾 Fine-tuned adapter saved locally to '{adapter_dir}'")


Generating train split: 0 examples [00:00, ? examples/s]

Dataset Loaded. Record Count: 500
⏳ Loading fresh base model 'google/gemma-4-E2B' in 4-bit...


config.json:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

...

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


⏳ Starting QLoRA fine-tuning...


Step,Training Loss
10,1.310630
20,0.155724
30,0.071797


✅ Fine-Tuning complete!
💾 Fine-tuned adapter saved locally to './fine_tuned_gemma_adapter'


## 🔬 Step 6: Evaluating the Fine-Tuned Model

Let's test our newly fine-tuned model (which is the frozen base model merged with our newly trained LoRA adapter).

Observe the output: the base model, which formerly autocompleted random text, has now successfully aligned to behave as a **sentiment classifier**, outputting the clean sentiment label!


In [5]:
from peft import PeftModel

# Let's run a test prompt directly on our current model state
test_reviews = [
    "Classify the sentiment: 'The software has a steep learning curve, but it is absolutely brilliant!'",
    "Classify the sentiment: 'Worst service ever, completely broken.'"
]

trainer.model.eval()

print("\n============================================================")
print("🚀 FINE-TUNED SPECIALIST MODEL INFERENCE RESULTS & ANALYSIS")
print("============================================================")

for pr in test_reviews:
    eval_messages = [{"role": "user", "content": pr}]
    formatted_prompt = base_tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
    inputs = base_tokenizer(formatted_prompt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id
        )

    prompt_len = inputs.input_ids.shape[1]
    generated_ids = outputs[0][prompt_len:]
    label = base_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    print(f"Prompt entered:  '{pr}'")
    print(f"Model Output:    '{label}'")
    print("-"*60)

print("\n💡 ANALYST EXPLANATION:")
print("This is a dramatic transformation! The raw Base Model, which previously")
print("rambled into high-entropy autocompletion loops, is now behaving as a")
print("perfectly constrained sentiment classification engine. By updating only")
print("a tiny fraction of parameters in the projection layers via QLoRA, we")
print("have steered the model's attention heads to output only our target labels")
print("('Positive' or 'Negative') with zero conversational fluff.")
print("============================================================")



🚀 FINE-TUNED SPECIALIST MODEL INFERENCE RESULTS & ANALYSIS


RuntimeError: Tensors must have same number of dimensions: got 2 and 3

### 🎮 Try Your Own Reviews on the Fine-Tuned Model!
Use the interactive form below to write your own reviews and see how our newly trained sentiment classification specialist classifies them.


In [ ]:
#@title 🚀 Interactive Fine-Tuned Model Playground { run: "auto" }
#@markdown Type your own review prompt below to test your fine-tuned sentiment classifier!

custom_prompt = "Classify the sentiment: 'The food was cold and the waiter was incredibly rude.'" #@param {type:"string"}

# Ensure fine-tuned model is in-memory
if 'trainer' in globals() and 'base_tokenizer' in globals():
    trainer.model.eval()
    eval_messages = [{"role": "user", "content": custom_prompt}]
    formatted_prompt = base_tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
    inputs = base_tokenizer(formatted_prompt, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id
        )

    prompt_len = inputs.input_ids.shape[1]
    response = base_tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()

    print("="*60)
    print("🚀 FINE-TUNED MODEL SPECIALIST RESPONSE:")
    print("="*60)
    print(response)
    print("="*60)
else:
    print("⚠️ Error: The Fine-tuned model or trainer is not in memory. Please complete the QLoRA training steps above.")


## ☁️ Step 7: Saving and Exporting Checkpoints to Google Cloud Storage (GCS)

To save these adapters permanently (since Colab local memory is ephemeral), you can easily serialize them and upload them directly to a Google Cloud Storage bucket. This allows your Cloud Run service to download them dynamically during deployment.

### GCS Integration Commands


In [ ]:
# 1. Authenticate your Google Cloud account within Colab
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated GCP User!")

# 2. Configure project variables
GCP_PROJECT_ID = "YOUR_GCP_PROJECT_ID" # Replace with your active project
GCS_BUCKET_NAME = "your-gemma-gcp-bucket" # Replace with your bucket

# Set active gcloud SDK project
!gcloud config set project {GCP_PROJECT_ID}

# 3. Create GCS bucket if not exists (using gcloud)
!gcloud storage buckets create gs://{GCS_BUCKET_NAME} --location=us-central1

# 4. Upload local fine-tuned adapter directory to GCS
!gcloud storage cp -r ./fine_tuned_gemma_adapter gs://{GCS_BUCKET_NAME}/gemma-4-adapters/
print(f"\n🎉 Successfully uploaded fine-tuned adapter weights to: gs://{GCS_BUCKET_NAME}/gemma-4-adapters/")


## 🤗 Step 8: Alternative Export Methods (Hugging Face Hub & Browser Download)

If you want to share your fine-tuned model with others so they can immediately test its behavior, or simply keep a local backup on your machine without using a GCP storage bucket, you can use these two highly convenient alternative export methods.

### 🛒 Option A: Share publicly on the Hugging Face Hub (Recommended)
Uploading your adapter directly to the Hugging Face Hub makes it incredibly easy for others to load and use. Anyone else running this notebook (or your local serving script) can simply reference your public model ID (e.g., `your_username/gemma-4-sentiment-adapter`) to run instant inference!


In [ ]:
#@title 🤗 Share adapter publicly on Hugging Face Hub { run: "manual" }
#@markdown Enter your Hugging Face username and desired repository name below.
#@markdown Make sure you logged in with a WRITE token in Step 1!

HF_USERNAME = "your_hf_username" #@param {type:"string"}
HF_REPO_NAME = "gemma-4-sentiment-adapter" #@param {type:"string"}

hub_model_id = f"{HF_USERNAME}/{HF_REPO_NAME}"

if HF_USERNAME == "your_hf_username":
    print("⚠️ Please enter your real Hugging Face username in the form-field above!")
else:
    print(f"⏳ Pushing adapter weights and tokenizer to Hugging Face Hub: {hub_model_id}...")
    try:
        # Push the PEFT adapter model and base tokenizer
        trainer.model.push_to_hub(hub_model_id, private=False)
        base_tokenizer.push_to_hub(hub_model_id, private=False)
        print(f"\n🎉 Successfully pushed to HF Hub! View it at: https://huggingface.co/{hub_model_id}")
        print("\n💡 Others can load and use your adapter directly using:")
        print(f"   PeftModel.from_pretrained(base_model, \"{hub_model_id}\")")
    except Exception as e:
        print(f"\n❌ Push failed: {e}")
        print("💡 Did you use a token with WRITE permissions during login in Step 1?")


### 📥 Option B: Download directly to your local machine

If you don't have a Google Cloud or Hugging Face account and just want a local copy of your fine-tuned weights, you can compress the adapter directory into a `.zip` archive and download it directly through your web browser.


In [ ]:
import os
import zipfile
from google.colab import files

zip_path = "./fine_tuned_gemma_adapter.zip"
adapter_dir = "./fine_tuned_gemma_adapter"

print("⏳ Compressing fine-tuned adapter folder...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(adapter_dir):
        for file in files_in_dir:
            file_path = os.path.join(root, file)
            # Maintain relative path inside zip
            arcname = os.path.relpath(file_path, os.path.dirname(adapter_dir))
            zipf.write(file_path, arcname)

print(f"✅ Compression complete! File size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
print("⏳ Starting browser download. Please allow popups if prompted...")
files.download(zip_path)
